# SAE-признаки: токсичность и отказ

Ноутбук показывает распределение токсичности комментариев и разложение активаций GPT-2 на SAE-фичи.

**Концепт** — изучаемый смысл, например оскорбление. **SAE-фича** — компонент внутренней активации модели; её численное значение показывает силу активации на токене.


## 1. Доменная область

| Концепт | Значение | Близкие, но отличающиеся случаи |
|---|---|---|
| **Токсичность** | Оскорбление, угроза или унижение человека либо группы в тексте. | Критика идеи, цитирование оскорбления, нейтральное упоминание группы. |
| **Refusal — отказ** | Полный или частичный отказ модели выполнять запрос, выраженный в ответе. | Извинение, нехватка знаний или предупреждение с последующим выполнением запроса. |

Опасность запроса, токсичность текста и отказ в ответе — разные характеристики. Опасный запрос бывает вежливым; безопасный запрос иногда вызывает ошибочный отказ.


## 2. Датасеты

| Датасет | Содержание и разметка | Связанные концепты |
|---|---|---|
| [Civil Comments](https://huggingface.co/datasets/google/civil_comments) | Комментарии с оценками `toxicity`, `insult`, `threat`, `identity_attack` и другими от 0 до 1. | Токсичность, оскорбление, угроза, нападка на группу. |
| [ToxiGen](https://huggingface.co/datasets/toxigen/toxigen-data) | Токсичные и нейтральные высказывания о группах; в поднаборе `annotated` есть человеческая оценка `toxicity_human`. | Неявная ненависть и стереотипы. |
| [WildGuardMix](https://huggingface.co/datasets/allenai/wildguardmix) | Запросы и ответы с отдельными метками опасности запроса, вредности ответа и отказа. | Опасная тема, вредный ответ, отказ. |
| [XSTest](https://github.com/paul-rottger/xstest) | Безопасные и небезопасные запросы; отказ определяется по ответу исследуемой модели. | Отказ на опасный запрос и ошибочный отказ на безопасный. |

Датасеты содержат разметку текстов, а соответствие между концептом и ID SAE-фичи определяется по активациям конкретной модели.


### Распределение токсичности до и после балансировки

**Оценка токсичности** — доля разметчиков, посчитавших комментарий токсичным: 0 — никто, 0,5 — половина, 1 — все. Здесь токсичными считаются комментарии с оценкой ≥ 0,5.

**Слева** показана исходная подвыборка Civil Comments. **Справа** — выборка с одинаковым числом токсичных и нетоксичных комментариев: до 500 каждого класса. Она получена случайным отбором без повторений; из более многочисленного класса исключается больше примеров.

Оба графика показывают долю комментариев в каждом диапазоне оценок на общей шкале. Синий цвет обозначает нетоксичные комментарии, красный — токсичные. Так выглядит балансировка классов при подготовке обучающей выборки. Модель здесь не обучается; проверочная выборка при такой схеме сохраняет исходное распределение классов.

In [ ]:
import os
from pathlib import Path

os.environ["HF_HOME"] = str(Path(".cache/huggingface").resolve())
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from datasets import load_dataset
from IPython.display import display

pio.renderers.default = "plotly_mimetype"
SEED, N = 42, 7000


def sample_dataset(name, split, config=None):
    rows = load_dataset(name, name=config, split=split, streaming=True)
    return pd.DataFrame(list(rows.shuffle(seed=SEED, buffer_size=10_000).take(N)))

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

civil = sample_dataset("google/civil_comments", "validation").drop_duplicates("text")
toxic = civil[civil.toxicity >= 0.5]
non_toxic = civil[civil.toxicity < 0.5]
per_class = min(500, len(toxic), len(non_toxic))
assert per_class > 0, "В подвыборке отсутствует один из классов"
balanced = pd.concat([part.sample(per_class, random_state=SEED)
                      for part in (non_toxic, toxic)], ignore_index=True)

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=["Исходная", "Баланс 50/50"])
for col, frame in enumerate((civil, balanced), start=1):
    for label, mask, color in (
        ("Нетоксичные (< 0,5)", frame.toxicity < 0.5, "#4C78A8"),
        ("Токсичные (≥ 0,5)", frame.toxicity >= 0.5, "#E45756")
    ):
        counts, edges = np.histogram(frame.loc[mask, "toxicity"], bins=np.linspace(0, 1, 21))
        fig.add_trace(go.Bar(x=(edges[:-1] + edges[1:]) / 2,
                            y=100 * counts / len(frame), width=0.05,
                            name=label, marker_color=color,
                            showlegend=False), row=1, col=col)
fig.update_layout(template="simple_white", barmode="stack", height=350,
                  showlegend=False, margin=dict(t=40, b=45, l=55, r=15))
fig.update_xaxes(title_text="Токсичность", range=[0, 1])
fig.update_yaxes(title_text="Комментарии, %", col=1)
fig.show(config={"displayModeBar": False})

## 3. Фичи и концепты

| Предполагаемый смысл фичи | Пример проявления концепта | Отличающийся контекст |
|---|---|---|
| **Оскорбление** | Унижение собеседника: «Ты идиот. | Критика вывода без унижения человека. |
| **Угроза** | Угроза причинить вред. | Рассказ об угрозе или её осуждение. |
| **Нападка на группу** | Унижение людей из-за принадлежности к группе. | Нейтральное упоминание той же группы. |
| **Отказ** | Ответ с отказом выполнить запрос. | Извинение, после которого запрос выполнен. |

Таблица описывает смысловые гипотезы. Один концепт может проявляться в нескольких фичах, а одна фича — реагировать на несколько смыслов. ID обозначает фичу в конкретных модели, слое и SAE.


### Разложение активаций

Используются **GPT-2 Small** и [готовый SAE](https://huggingface.co/jbloom/GPT2-Small-SAEs-Reformatted) для входа в блок 8: **768 координат активации → 24 576 SAE-фич → 768 координат восстановления**.

Восстановление складывается из **смещения и суммы вкладов фич**. Каждый вклад — сила активации фичи, умноженная на её направление. Разница между исходной и восстановленной активацией составляет ошибку восстановления.

Тепловая карта показывает 10 фич с наибольшими пиковыми активациями на данном тексте. По горизонтали расположены токены, по вертикали — ID фич, цвет обозначает силу активации. Ниже выводятся среднее число активных фич на токен и относительная ошибка восстановления. Начальный служебный токен BOS исключён.


In [ ]:
import torch
from sae_lens import SAE, HookedSAETransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb", sae_id="blocks.8.hook_resid_pre", device=DEVICE
).eval()
model = HookedSAETransformer.from_pretrained_no_processing(
    sae.cfg.metadata.model_name, device=DEVICE,
    **(sae.cfg.metadata.model_from_pretrained_kwargs or {})
).eval()
HOOK = sae.cfg.metadata.hook_name
MAX_TOKENS = 64


@torch.inference_mode()
def decompose(text):
    tokens = model.to_tokens(text)[:, :MAX_TOKENS]
    _, cache = model.run_with_cache(tokens, names_filter=[HOOK], return_type=None)
    h = cache[HOOK].to(sae.W_dec.dtype)
    z = sae.encode(h)
    restored = sae.decode(z)
    return tokens, h[0, 1:].cpu(), z[0, 1:].cpu(), restored[0, 1:].cpu()

In [ ]:
TEXT = "You are a wonderful person. You are a complete idiot."
tokens, h, z, restored = decompose(TEXT)
top = z.amax(dim=0).topk(10).indices
# Номер позиции различает одинаковые токены на графике.
token_names = [f"{i}: {model.tokenizer.decode([int(t)])}"
               for i, t in enumerate(tokens[0, 1:].cpu())]
heatmap = px.imshow(z[:, top].T.numpy(), x=token_names, y=[str(j) for j in top.tolist()],
          aspect="auto", template="simple_white",
          labels={"x": "Токен", "y": "ID фичи", "color": "Активация"})
heatmap.update_layout(margin=dict(t=15, b=50, l=65, r=30))
heatmap.show(config={"displayModeBar": False})

print(f"В среднем активно {(z > 0).sum(1).float().mean():.1f} из {z.shape[1]} фич на токен")
print(f"Относительная ошибка восстановления: {(h - restored).norm() / h.norm():.3f}")

В среднем активно 53.5 из 24576 фич на токен
Относительная ошибка восстановления: 0.254
